# 06 Train GraphEdgeClassifier - Class Weight

Notebook ini dibuat untuk tujuan research NFT wash trading detection dengan **GraphEdgeClassifier + class weight**.

Fokus utama notebook ini:
- memakai split baru: `train_1_50.csv`, `val_temporal.csv`, `test_temporal.csv`
- validation dipakai untuk model selection dan threshold tuning
- test hanya dipakai untuk final evaluation
- **menghapus label leakage** dari rule-based columns
- menambahkan temporal/pair features yang tidak memakai label
- class weight dibuat lebih stabil memakai `sqrt/capped pos_weight`
- evaluasi diarahkan ke high-recall candidate detection, FP budget, top-k ranking, dan lift vs random

In [1]:
import json
import random
import copy
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

device(type='cpu')

In [2]:
# =========================
# PATH CONFIG
# =========================

DATA_DIR = Path("../data/processed")
OUTPUT_METRICS_DIR = Path("../outputs/metrics")
OUTPUT_MODELS_DIR = Path("../outputs/models")

OUTPUT_METRICS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_MODELS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "train_1_50.csv"
VAL_PATH   = DATA_DIR / "val_temporal.csv"
TEST_PATH  = DATA_DIR / "test_temporal.csv"

for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    print(p, "exists=", p.exists())
    if not p.exists():
        raise FileNotFoundError(f"File tidak ditemukan: {p}. Jalankan dulu notebook split terbaru.")

..\data\processed\train_1_50.csv exists= True
..\data\processed\val_temporal.csv exists= True
..\data\processed\test_temporal.csv exists= True


In [3]:
# =========================
# LOAD SPLIT DATA
# =========================

train_df = pd.read_csv(TRAIN_PATH, low_memory=False)
val_df   = pd.read_csv(VAL_PATH, low_memory=False)
test_df  = pd.read_csv(TEST_PATH, low_memory=False)

print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)
print("Columns:")
print(train_df.columns.tolist())

Train: (35802, 25)
Val  : (149079, 25)
Test : (298160, 25)
Columns:
['transaction_hash', 'block_number', 'timestamp', 'nft_address', 'token_id', 'from_address', 'to_address', 'transaction_value', 'mint_timestamp', 'transfers_out_from', 'transfers_in_from', 'transfers_out_to', 'transfers_in_to', 'num_transitions', 'timestamp_dt', 'mint_timestamp_dt', 'month', 'is_wash_trading', 'rule_self_trade', 'rule_seller_buyback', 'rule_multi_hop_cycle', 'rule_high_pair_count', 'wash_score', 'confidence_category', 'label_final']


In [4]:
# =========================
# COLUMN CONFIG
# =========================

SOURCE_COL = "from_address"
TARGET_COL = "to_address"
LABEL_COL = "label_final"
TIMESTAMP_COL = "timestamp"
VALUE_COL = "transaction_value" if "transaction_value" in train_df.columns else None
TOKEN_COL = "token_id" if "token_id" in train_df.columns else None
NFT_COL = "nft_address" if "nft_address" in train_df.columns else None
MINT_TIMESTAMP_COL = "mint_timestamp" if "mint_timestamp" in train_df.columns else None

required_cols = [SOURCE_COL, TARGET_COL, LABEL_COL, TIMESTAMP_COL]
missing = [c for c in required_cols if c not in train_df.columns]
if missing:
    raise ValueError(f"Kolom wajib tidak ditemukan: {missing}")

for df in [train_df, val_df, test_df]:
    df[TIMESTAMP_COL] = pd.to_numeric(df[TIMESTAMP_COL], errors="coerce")
    df[LABEL_COL] = pd.to_numeric(df[LABEL_COL], errors="coerce").fillna(0).astype(int)
    if VALUE_COL is not None:
        df[VALUE_COL] = pd.to_numeric(df[VALUE_COL], errors="coerce").fillna(0)
    if MINT_TIMESTAMP_COL is not None:
        df[MINT_TIMESTAMP_COL] = pd.to_numeric(df[MINT_TIMESTAMP_COL], errors="coerce")

train_df = train_df.sort_values(TIMESTAMP_COL).reset_index(drop=True)
val_df   = val_df.sort_values(TIMESTAMP_COL).reset_index(drop=True)
test_df  = test_df.sort_values(TIMESTAMP_COL).reset_index(drop=True)

for name, df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    print(name, df[LABEL_COL].value_counts(dropna=False).to_dict(), "fraud_rate=", float(df[LABEL_COL].mean()))
    print("  timestamp:", df[TIMESTAMP_COL].min(), "->", df[TIMESTAMP_COL].max())

Train {0: 35100, 1: 702} fraud_rate= 0.0196078431372549
  timestamp: 1622507092 -> 1629695254
Val {0: 149027, 1: 52} fraud_rate= 0.0003488083499352692
  timestamp: 1629695273 -> 1629978053
Test {0: 298096, 1: 64} fraud_rate= 0.00021464985242822645
  timestamp: 1629978053 -> 1630454395


## Feature engineering tanpa label leakage

Fitur rule-based seperti `wash_score`, `rule_*`, `is_wash_trading`, dan `confidence_category` **tidak dipakai sebagai input model** karena kolom tersebut membentuk `label_final`.

Notebook ini menambahkan fitur temporal/pair berbasis riwayat masa lalu saja, misalnya jumlah transaksi wallet sebelumnya, jumlah interaksi pair sebelumnya, dan waktu sejak transaksi terakhir.

In [5]:
# =========================
# TEMPORAL / PAIR FEATURES - PAST ONLY, NO LABEL USAGE
# =========================

ENGINEERED_FEATURES = [
    "src_tx_count_past",
    "dst_tx_count_past",
    "src_to_dst_count_past",
    "dst_to_src_count_past",
    "pair_undirected_count_past",
    "token_tx_count_past",
    "nft_tx_count_past",
    "time_since_src_last",
    "time_since_dst_last",
    "log_time_since_src_last",
    "log_time_since_dst_last",
    "src_value_sum_past",
    "dst_value_sum_past",
    "pair_value_sum_past",
    "src_value_mean_past",
    "dst_value_mean_past",
    "pair_value_mean_past",
    "mint_age_seconds",
    "log_transaction_value",
]


def init_temporal_state():
    return {
        "wallet_count": defaultdict(int),
        "wallet_last_time": {},
        "wallet_value_sum": defaultdict(float),
        "pair_directed_count": defaultdict(int),
        "pair_undirected_count": defaultdict(int),
        "pair_value_sum": defaultdict(float),
        "token_count": defaultdict(int),
        "nft_count": defaultdict(int),
    }


def add_past_temporal_features(df, state=None, update_state=True):
    """Tambah fitur berbasis event masa lalu.

    Untuk train: state kosong, update per transaksi.
    Untuk val: state hasil train, update per transaksi val.
    Untuk test: state hasil train+val, update per transaksi test.

    Tidak ada label yang digunakan.
    """
    if state is None:
        state = init_temporal_state()
    df = df.sort_values(TIMESTAMP_COL).reset_index(drop=True).copy()

    rows = {name: [] for name in ENGINEERED_FEATURES}

    for _, row in df.iterrows():
        src = str(row[SOURCE_COL]) if pd.notna(row[SOURCE_COL]) else "UNKNOWN_WALLET"
        dst = str(row[TARGET_COL]) if pd.notna(row[TARGET_COL]) else "UNKNOWN_WALLET"
        ts = float(row[TIMESTAMP_COL]) if pd.notna(row[TIMESTAMP_COL]) else 0.0
        value = float(row[VALUE_COL]) if VALUE_COL is not None and pd.notna(row[VALUE_COL]) else 0.0
        token = str(row[TOKEN_COL]) if TOKEN_COL is not None and pd.notna(row[TOKEN_COL]) else "UNKNOWN_TOKEN"
        nft = str(row[NFT_COL]) if NFT_COL is not None and pd.notna(row[NFT_COL]) else "UNKNOWN_NFT"
        pair_dir = (src, dst)
        pair_rev = (dst, src)
        pair_undir = tuple(sorted([src, dst]))

        src_count = state["wallet_count"][src]
        dst_count = state["wallet_count"][dst]
        pair_dir_count = state["pair_directed_count"][pair_dir]
        pair_rev_count = state["pair_directed_count"][pair_rev]
        pair_undir_count = state["pair_undirected_count"][pair_undir]
        token_count = state["token_count"][token]
        nft_count = state["nft_count"][nft]

        src_last = state["wallet_last_time"].get(src, np.nan)
        dst_last = state["wallet_last_time"].get(dst, np.nan)
        time_since_src = ts - src_last if pd.notna(src_last) else 0.0
        time_since_dst = ts - dst_last if pd.notna(dst_last) else 0.0
        time_since_src = max(float(time_since_src), 0.0)
        time_since_dst = max(float(time_since_dst), 0.0)

        src_value_sum = state["wallet_value_sum"][src]
        dst_value_sum = state["wallet_value_sum"][dst]
        pair_value_sum = state["pair_value_sum"][pair_undir]

        src_value_mean = src_value_sum / src_count if src_count > 0 else 0.0
        dst_value_mean = dst_value_sum / dst_count if dst_count > 0 else 0.0
        pair_value_mean = pair_value_sum / pair_undir_count if pair_undir_count > 0 else 0.0

        if MINT_TIMESTAMP_COL is not None and pd.notna(row[MINT_TIMESTAMP_COL]):
            mint_age = max(ts - float(row[MINT_TIMESTAMP_COL]), 0.0)
        else:
            mint_age = 0.0

        rows["src_tx_count_past"].append(src_count)
        rows["dst_tx_count_past"].append(dst_count)
        rows["src_to_dst_count_past"].append(pair_dir_count)
        rows["dst_to_src_count_past"].append(pair_rev_count)
        rows["pair_undirected_count_past"].append(pair_undir_count)
        rows["token_tx_count_past"].append(token_count)
        rows["nft_tx_count_past"].append(nft_count)
        rows["time_since_src_last"].append(time_since_src)
        rows["time_since_dst_last"].append(time_since_dst)
        rows["log_time_since_src_last"].append(np.log1p(time_since_src))
        rows["log_time_since_dst_last"].append(np.log1p(time_since_dst))
        rows["src_value_sum_past"].append(src_value_sum)
        rows["dst_value_sum_past"].append(dst_value_sum)
        rows["pair_value_sum_past"].append(pair_value_sum)
        rows["src_value_mean_past"].append(src_value_mean)
        rows["dst_value_mean_past"].append(dst_value_mean)
        rows["pair_value_mean_past"].append(pair_value_mean)
        rows["mint_age_seconds"].append(mint_age)
        rows["log_transaction_value"].append(np.log1p(max(value, 0.0)))

        if update_state:
            state["wallet_count"][src] += 1
            state["wallet_count"][dst] += 1
            state["wallet_last_time"][src] = ts
            state["wallet_last_time"][dst] = ts
            state["wallet_value_sum"][src] += value
            state["wallet_value_sum"][dst] += value
            state["pair_directed_count"][pair_dir] += 1
            state["pair_undirected_count"][pair_undir] += 1
            state["pair_value_sum"][pair_undir] += value
            state["token_count"][token] += 1
            state["nft_count"][nft] += 1

    for name, values in rows.items():
        df[name] = np.asarray(values, dtype=np.float32)

    return df, state

state0 = init_temporal_state()
train_df, state_after_train = add_past_temporal_features(train_df, state0, update_state=True)
val_df, state_after_val = add_past_temporal_features(val_df, copy.deepcopy(state_after_train), update_state=True)
test_df, _ = add_past_temporal_features(test_df, copy.deepcopy(state_after_val), update_state=True)

print("Engineered features added:", len(ENGINEERED_FEATURES))
train_df[ENGINEERED_FEATURES].head()

Engineered features added: 19


,src_tx_count_past,dst_tx_count_past,src_to_dst_count_past,dst_to_src_count_past,pair_undirected_count_past,token_tx_count_past,nft_tx_count_past,time_since_src_last,time_since_dst_last,log_time_since_src_last,log_time_since_dst_last,src_value_sum_past,dst_value_sum_past,pair_value_sum_past,src_value_mean_past,dst_value_mean_past,pair_value_mean_past,mint_age_seconds,log_transaction_value
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,37.321579
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3287582.0,37.246826
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,38.450798
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,34.563469
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,38.227657


In [6]:
# =========================
# NODE MAPPING
# =========================
# Mapping dibuat dari train+val+test supaya wallet baru di validation/test tetap punya ID.
# Ini hanya memakai daftar wallet, bukan label atau fitur masa depan untuk training.

all_nodes = pd.concat([
    train_df[SOURCE_COL], train_df[TARGET_COL],
    val_df[SOURCE_COL], val_df[TARGET_COL],
    test_df[SOURCE_COL], test_df[TARGET_COL],
], axis=0).astype(str).fillna("UNKNOWN_WALLET").unique()

node_to_id = {node: idx for idx, node in enumerate(all_nodes)}
num_nodes = len(node_to_id)
print("Num nodes:", num_nodes)


def map_nodes(df):
    src = df[SOURCE_COL].astype(str).fillna("UNKNOWN_WALLET").map(node_to_id)
    dst = df[TARGET_COL].astype(str).fillna("UNKNOWN_WALLET").map(node_to_id)
    if src.isna().any() or dst.isna().any():
        raise ValueError(f"Ada node yang tidak ada di mapping. missing_src={src.isna().sum()}, missing_dst={dst.isna().sum()}")
    return src.astype(np.int64).values, dst.astype(np.int64).values

Num nodes: 126505


In [7]:
# =========================
# FEATURE SELECTION - STRICT NO LEAKAGE
# =========================

leakage_cols = {
    # target / direct label leakage
    LABEL_COL,
    "is_wash_trading",

    # rule-based columns used to create label_final
    "rule_self_trade",
    "rule_seller_buyback",
    "rule_multi_hop_cycle",
    "rule_high_pair_count",
    "wash_score",
    "confidence_category",
}

id_time_cols = {
    SOURCE_COL,
    TARGET_COL,
    TIMESTAMP_COL,
    "transaction_hash",
    "nft_address",
    "token_id",
    "timestamp_dt",
    "mint_timestamp_dt",
}

# Jangan pakai timestamp mentah sebagai feature, tapi boleh pakai engineered time deltas.
exclude_cols = leakage_cols | id_time_cols

numeric_cols = train_df.select_dtypes(include=[np.number, "bool"]).columns.tolist()
feature_cols = [c for c in numeric_cols if c not in exclude_cols]

# Pastikan engineered features masuk.
for c in ENGINEERED_FEATURES:
    if c not in feature_cols and c in train_df.columns:
        feature_cols.append(c)

# Remove duplicates while preserving order.
feature_cols = list(dict.fromkeys(feature_cols))

leaked = sorted(set(feature_cols) & leakage_cols)
if leaked:
    raise ValueError(f"Masih ada leakage columns di feature_cols: {leaked}")
if len(feature_cols) == 0:
    raise ValueError("Tidak ada numeric feature yang tersedia.")

print("Num feature cols:", len(feature_cols))
print("Features used:")
for c in feature_cols:
    print("-", c)

print("Excluded leakage columns:")
for c in sorted(leakage_cols):
    print("-", c)

Num feature cols: 28
Features used:
- block_number
- transaction_value
- mint_timestamp
- transfers_out_from
- transfers_in_from
- transfers_out_to
- transfers_in_to
- num_transitions
- month
- src_tx_count_past
- dst_tx_count_past
- src_to_dst_count_past
- dst_to_src_count_past
- pair_undirected_count_past
- token_tx_count_past
- nft_tx_count_past
- time_since_src_last
- time_since_dst_last
- log_time_since_src_last
- log_time_since_dst_last
- src_value_sum_past
- dst_value_sum_past
- pair_value_sum_past
- src_value_mean_past
- dst_value_mean_past
- pair_value_mean_past
- mint_age_seconds
- log_transaction_value
Excluded leakage columns:
- confidence_category
- is_wash_trading
- label_final
- rule_high_pair_count
- rule_multi_hop_cycle
- rule_self_trade
- rule_seller_buyback
- wash_score


In [8]:
# =========================
# FEATURE SCALING
# =========================
# Fit scaler hanya di train, lalu transform val/test.

def make_feature_matrix(df, feature_cols):
    return (
        df[feature_cols]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
        .astype(np.float32)
        .values
    )

scaler = StandardScaler()
X_train_raw = make_feature_matrix(train_df, feature_cols)
X_val_raw = make_feature_matrix(val_df, feature_cols)
X_test_raw = make_feature_matrix(test_df, feature_cols)

X_train = scaler.fit_transform(X_train_raw).astype(np.float32)
X_val = scaler.transform(X_val_raw).astype(np.float32)
X_test = scaler.transform(X_test_raw).astype(np.float32)

print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)
print("Any NaN train:", np.isnan(X_train).any())

X_train: (35802, 28)
X_val  : (149079, 28)
X_test : (298160, 28)
Any NaN train: False


In [9]:
# =========================
# DATASET & DATALOADER
# =========================

class EdgeDataset(Dataset):
    def __init__(self, df, features):
        self.source, self.target = map_nodes(df)
        self.edge_features = features
        self.labels = df[LABEL_COL].astype(np.float32).values

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.source[idx], dtype=torch.long),
            torch.tensor(self.target[idx], dtype=torch.long),
            torch.tensor(self.edge_features[idx], dtype=torch.float32),
            torch.tensor(self.labels[idx], dtype=torch.float32),
        )

BATCH_SIZE = 2048

train_dataset = EdgeDataset(train_df, X_train)
val_dataset = EdgeDataset(val_df, X_val)
test_dataset = EdgeDataset(test_df, X_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

edge_feat_dim = X_train.shape[1]
print("edge_feat_dim:", edge_feat_dim)

edge_feat_dim: 28


In [10]:
# =========================
# MODEL
# =========================

class GraphEdgeClassifier(nn.Module):
    def __init__(self, num_nodes, edge_feat_dim, embedding_dim=64, hidden_dim=128, dropout=0.35):
        super().__init__()
        self.node_embedding = nn.Embedding(num_nodes, embedding_dim)
        input_dim = embedding_dim * 2 + edge_feat_dim
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, source, target, edge_feat):
        src_emb = self.node_embedding(source)
        dst_emb = self.node_embedding(target)
        x = torch.cat([src_emb, dst_emb, edge_feat], dim=1)
        return self.mlp(x).squeeze(-1)

model = GraphEdgeClassifier(
    num_nodes=num_nodes,
    edge_feat_dim=edge_feat_dim,
    embedding_dim=64,
    hidden_dim=128,
    dropout=0.35,
).to(DEVICE)

model

GraphEdgeClassifier(
  (node_embedding): Embedding(126505, 64)
  (mlp): Sequential(
    (0): Linear(in_features=156, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.35, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.35, inplace=False)
    (8): Linear(in_features=64, out_features=1, bias=True)
  )
)

In [11]:
# =========================
# CLASS WEIGHT LOSS - STABLE VERSION
# =========================
# Karena train sudah di-undersampling 1:50, raw pos_weight=50 sering terlalu agresif.
# Untuk research ini, gunakan sqrt/capped pos_weight agar model tetap peduli fraud
# tanpa membuat semua normal menjadi fraud.

labels = train_df[LABEL_COL].astype(int).values
num_positive = int(np.sum(labels == 1))
num_negative = int(np.sum(labels == 0))
raw_pos_weight = num_negative / max(num_positive, 1)

POS_WEIGHT_MODE = "sqrt_capped"  # options: "raw", "sqrt", "sqrt_capped", "manual"
MANUAL_POS_WEIGHT = 10.0
POS_WEIGHT_CAP = 10.0

if POS_WEIGHT_MODE == "raw":
    pos_weight_value = raw_pos_weight
elif POS_WEIGHT_MODE == "sqrt":
    pos_weight_value = np.sqrt(raw_pos_weight)
elif POS_WEIGHT_MODE == "sqrt_capped":
    pos_weight_value = min(np.sqrt(raw_pos_weight), POS_WEIGHT_CAP)
elif POS_WEIGHT_MODE == "manual":
    pos_weight_value = MANUAL_POS_WEIGHT
else:
    raise ValueError("Unknown POS_WEIGHT_MODE")

print("Train Positive:", num_positive)
print("Train Negative:", num_negative)
print("raw_pos_weight:", raw_pos_weight)
print("used_pos_weight:", pos_weight_value)

pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32, device=DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

Train Positive: 702
Train Negative: 35100
raw_pos_weight: 50.0
used_pos_weight: 7.0710678118654755


In [12]:
# =========================
# TRAIN / EVAL HELPERS
# =========================

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    n = 0
    for source, target, edge_feat, label in loader:
        source = source.to(DEVICE)
        target = target.to(DEVICE)
        edge_feat = edge_feat.to(DEVICE)
        label = label.to(DEVICE)

        optimizer.zero_grad()
        logits = model(source, target, edge_feat)
        loss = criterion(logits, label)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        bs = label.size(0)
        total_loss += float(loss.item()) * bs
        n += bs
    return total_loss / max(n, 1)


@torch.no_grad()
def predict_proba(model, loader):
    model.eval()
    all_probs = []
    all_labels = []
    for source, target, edge_feat, label in loader:
        source = source.to(DEVICE)
        target = target.to(DEVICE)
        edge_feat = edge_feat.to(DEVICE)
        logits = model(source, target, edge_feat)
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        all_probs.append(probs)
        all_labels.append(label.numpy())
    return np.concatenate(all_labels), np.concatenate(all_probs)


def safe_auc(labels, probs, fn):
    try:
        return float(fn(labels, probs))
    except ValueError:
        return float("nan")


def compute_metrics(labels, probs, threshold=0.5):
    preds = (probs >= threshold).astype(int)
    cm = confusion_matrix(labels, preds, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return {
        "accuracy": float(accuracy_score(labels, preds)),
        "precision": float(precision_score(labels, preds, zero_division=0)),
        "recall": float(recall_score(labels, preds, zero_division=0)),
        "f1": float(f1_score(labels, preds, zero_division=0)),
        "roc_auc": safe_auc(labels, probs, roc_auc_score),
        "pr_auc": safe_auc(labels, probs, average_precision_score),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

In [13]:
# =========================
# TRAINING WITH VALIDATION MODEL SELECTION
# =========================

EPOCHS = 30
PATIENCE = 6
BEST_MODEL_PATH = OUTPUT_MODELS_DIR / "graph_class_weight_upgrade_no_leakage_best.pt"

history = []
best_score = -1
best_epoch = 0
no_improve = 0

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion)

    val_labels, val_probs = predict_proba(model, val_loader)
    val_m = compute_metrics(val_labels, val_probs, threshold=0.5)

    # Model selection pakai PR-AUC karena fraud sangat imbalance.
    score = val_m["pr_auc"] if not np.isnan(val_m["pr_auc"]) else -1

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_accuracy@0.5": val_m["accuracy"],
        "val_precision@0.5": val_m["precision"],
        "val_recall@0.5": val_m["recall"],
        "val_f1@0.5": val_m["f1"],
        "val_roc_auc": val_m["roc_auc"],
        "val_pr_auc": val_m["pr_auc"],
    }
    history.append(row)
    print(row)

    if score > best_score:
        best_score = score
        best_epoch = epoch
        no_improve = 0
        torch.save(model.state_dict(), BEST_MODEL_PATH)
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch}. Best epoch: {best_epoch}")
            break

history_df = pd.DataFrame(history)
history_df.to_csv(OUTPUT_METRICS_DIR / "graph_class_weight_upgrade_no_leakage_history.csv", index=False)
history_df

{'epoch': 1, 'train_loss': 0.5486633750165176, 'val_accuracy@0.5': 0.7925059867586984, 'val_precision@0.5': 0.0010663392251268296, 'val_recall@0.5': 0.6346153846153846, 'val_f1@0.5': 0.0021291009387399596, 'val_roc_auc': 0.7725574121571155, 'val_pr_auc': 0.002433928680167921}
{'epoch': 2, 'train_loss': 0.39635157638085994, 'val_accuracy@0.5': 0.7414256870518315, 'val_precision@0.5': 0.0008816512809874495, 'val_recall@0.5': 0.6538461538461539, 'val_f1@0.5': 0.0017609281126993992, 'val_roc_auc': 0.777469854455904, 'val_pr_auc': 0.002547859667445011}
{'epoch': 3, 'train_loss': 0.31621857531962666, 'val_accuracy@0.5': 0.7066857169688554, 'val_precision@0.5': 0.0008000914390216024, 'val_recall@0.5': 0.6730769230769231, 'val_f1@0.5': 0.0015982829874192296, 'val_roc_auc': 0.7770658233846113, 'val_pr_auc': 0.00253254521786689}
{'epoch': 4, 'train_loss': 0.2634713231241974, 'val_accuracy@0.5': 0.6810080561313129, 'val_precision@0.5': 0.0007566999474513925, 'val_recall@0.5': 0.6923076923076923, 

,epoch,train_loss,val_accuracy@0.5,val_precision@0.5,val_recall@0.5,val_f1@0.5,val_roc_auc,val_pr_auc
0,1,0.548663,0.792506,0.001066,0.634615,0.002129,0.772557,0.002434
1,2,0.396352,0.741426,0.000882,0.653846,0.001761,0.777470,0.002548
2,3,0.316219,0.706686,0.000800,0.673077,0.001598,0.777066,0.002533
3,4,0.263471,0.681008,0.000757,0.692308,0.001512,0.780340,0.002850
4,5,0.231318,0.676916,0.000768,0.711538,0.001534,0.778763,0.003302
5,6,0.205726,0.656994,0.000723,0.711538,0.001445,0.779265,0.003642
6,7,0.187141,0.678345,0.000771,0.711538,0.001541,0.778851,0.004718
7,8,0.167691,0.654318,0.000718,0.711538,0.001434,0.773602,0.004564
8,9,0.154555,0.666036,0.000743,0.711538,0.001484,0.774343,0.005481
9,10,0.143374,0.657759,0.000725,0.711538,0.001448,0.770988,0.006721


In [14]:
# =========================
# VALIDATION THRESHOLD SEARCH
# =========================

def build_threshold_table(labels, probs):
    quantile_thresholds = np.unique(np.quantile(probs, np.linspace(0, 1, 1001)))
    fixed_thresholds = np.array([0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5])
    thresholds = np.unique(np.concatenate([quantile_thresholds, fixed_thresholds]))

    rows = []
    for t in thresholds:
        m = compute_metrics(labels, probs, threshold=float(t))
        rows.append({"threshold": float(t), **m})
    return pd.DataFrame(rows).sort_values("threshold").reset_index(drop=True)


def select_threshold(df, strategy="fp_budget_then_recall", recall_target=0.6, max_fp_rate=0.05):
    max_fp = int(max_fp_rate * (df["tn"] + df["fp"]).max())

    if strategy == "best_f1":
        best = df.sort_values(["f1", "recall", "precision"], ascending=False).iloc[0]
    elif strategy == "recall_target_min_fp":
        cand = df[df["recall"] >= recall_target].copy()
        if len(cand) == 0:
            best = df.sort_values(["recall", "f1"], ascending=False).iloc[0]
        else:
            best = cand.sort_values(["fp", "precision", "f1"], ascending=[True, False, False]).iloc[0]
    elif strategy == "fp_budget_then_recall":
        cand = df[df["fp"] <= max_fp].copy()
        if len(cand) == 0:
            best = df.sort_values(["fp", "recall", "f1"], ascending=[True, False, False]).iloc[0]
        else:
            # Dalam budget FP, pilih recall tertinggi, lalu precision/F1 tertinggi.
            best = cand.sort_values(["recall", "precision", "f1"], ascending=False).iloc[0]
    else:
        raise ValueError("Unknown threshold strategy")

    return float(best["threshold"]), best.to_dict(), max_fp

model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))
val_labels, val_probs = predict_proba(model, val_loader)
val_threshold_table = build_threshold_table(val_labels, val_probs)

THRESHOLD_STRATEGY = "fp_budget_then_recall"
RECALL_TARGET = 0.60
MAX_FP_RATE_ON_VAL_NORMAL = 0.05  # 5% normal validation boleh menjadi alert candidate.

BEST_THRESHOLD, best_threshold_row, val_fp_budget = select_threshold(
    val_threshold_table,
    strategy=THRESHOLD_STRATEGY,
    recall_target=RECALL_TARGET,
    max_fp_rate=MAX_FP_RATE_ON_VAL_NORMAL,
)

print("Best epoch:", best_epoch)
print("Best validation PR-AUC:", best_score)
print("Threshold strategy:", THRESHOLD_STRATEGY)
print("Validation FP budget:", val_fp_budget)
print("Best threshold:", BEST_THRESHOLD)
print(json.dumps(best_threshold_row, indent=2))

val_threshold_table.to_csv(OUTPUT_METRICS_DIR / "graph_class_weight_upgrade_no_leakage_thresholds_validation.csv", index=False)
val_threshold_table.sort_values(["f1", "recall"], ascending=False).head(20)

Best epoch: 27
Best validation PR-AUC: 0.053153929825627895
Threshold strategy: fp_budget_then_recall
Validation FP budget: 7451
Best threshold: 0.9010158498287203
{
  "threshold": 0.9010158498287203,
  "accuracy": 0.9510259660985115,
  "precision": 0.0038329911019849418,
  "recall": 0.5384615384615384,
  "f1": 0.007611798287345386,
  "roc_auc": 0.7350329134988962,
  "pr_auc": 0.053153929825627895,
  "tn": 141750.0,
  "fp": 7277.0,
  "fn": 24.0,
  "tp": 28.0
}


,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc,tn,fp,fn,tp
1009,0.999401,0.998873,0.113333,0.326923,0.168317,0.735033,0.053154,148894,133,35,17
1008,0.997792,0.997941,0.073579,0.423077,0.125356,0.735033,0.053154,148750,277,30,22
1007,0.996423,0.996968,0.053571,0.461538,0.096000,0.735033,0.053154,148603,424,28,24
1006,0.995051,0.995982,0.041876,0.480769,0.077042,0.735033,0.053154,148455,572,27,25
1005,0.993389,0.994983,0.033512,0.480769,0.062657,0.735033,0.053154,148306,721,27,25
1004,0.991898,0.993983,0.027933,0.480769,0.052798,0.735033,0.053154,148157,870,27,25
1003,0.990381,0.992984,0.023946,0.480769,0.045620,0.735033,0.053154,148008,1019,27,25
1002,0.989025,0.991984,0.020956,0.480769,0.040161,0.735033,0.053154,147859,1168,27,25
1001,0.987132,0.990985,0.018629,0.480769,0.035868,0.735033,0.053154,147710,1317,27,25
1000,0.985339,0.989985,0.016767,0.480769,0.032404,0.735033,0.053154,147561,1466,27,25


In [15]:
# =========================
# FINAL TEST EVALUATION
# =========================

test_labels, test_probs = predict_proba(model, test_loader)
final_m = compute_metrics(test_labels, test_probs, threshold=BEST_THRESHOLD)

final_result = {
    "model": "graph_class_weight_upgrade_no_leakage",
    "best_epoch_from_validation": int(best_epoch),
    "pos_weight_mode": POS_WEIGHT_MODE,
    "raw_pos_weight": float(raw_pos_weight),
    "used_pos_weight": float(pos_weight_value),
    "threshold_strategy": THRESHOLD_STRATEGY,
    "best_threshold_from_validation": float(BEST_THRESHOLD),
    "validation_fp_budget": int(val_fp_budget),
    **final_m,
}

pd.DataFrame([final_result]).to_csv(OUTPUT_METRICS_DIR / "graph_class_weight_upgrade_no_leakage_final.csv", index=False)

print(json.dumps(final_result, indent=2))
print("Confusion Matrix [[TN, FP], [FN, TP]]:")
print(np.array([[final_m["tn"], final_m["fp"]], [final_m["fn"], final_m["tp"]]]))

{
  "model": "graph_class_weight_upgrade_no_leakage",
  "best_epoch_from_validation": 27,
  "pos_weight_mode": "sqrt_capped",
  "raw_pos_weight": 50.0,
  "used_pos_weight": 7.0710678118654755,
  "threshold_strategy": "fp_budget_then_recall",
  "best_threshold_from_validation": 0.9010158498287203,
  "validation_fp_budget": 7451,
  "accuracy": 0.49525757982291385,
  "precision": 0.00031224256596954636,
  "recall": 0.734375,
  "f1": 0.0006242197253433209,
  "roc_auc": 0.7363907359122565,
  "pr_auc": 0.015742232055127998,
  "tn": 147619,
  "fp": 150477,
  "fn": 17,
  "tp": 47
}
Confusion Matrix [[TN, FP], [FN, TP]]:
[[147619 150477]
 [    17     47]]


In [16]:
# =========================
# TEST THRESHOLD TABLE - ANALYSIS ONLY
# =========================
# Test threshold table hanya untuk analisis laporan. Jangan memilih threshold dari sini.

test_threshold_rows = []
for t in sorted(set([0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, BEST_THRESHOLD])):
    test_threshold_rows.append({"threshold": float(t), **compute_metrics(test_labels, test_probs, threshold=float(t))})

test_threshold_df = pd.DataFrame(test_threshold_rows)
test_threshold_df.to_csv(OUTPUT_METRICS_DIR / "graph_class_weight_upgrade_no_leakage_thresholds_test_analysis.csv", index=False)
test_threshold_df

,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc,tn,fp,fn,tp
0,0.001000,0.079162,0.000215,0.921875,0.000430,0.736391,0.015742,23544,274552,5,59
1,0.005000,0.108938,0.000218,0.906250,0.000436,0.736391,0.015742,32423,265673,6,58
2,0.010000,0.125490,0.000222,0.906250,0.000445,0.736391,0.015742,37358,260738,6,58
3,0.020000,0.144037,0.000227,0.906250,0.000454,0.736391,0.015742,42888,255208,6,58
4,0.050000,0.172763,0.000231,0.890625,0.000462,0.736391,0.015742,51454,246642,7,57
5,0.100000,0.200067,0.000231,0.859375,0.000461,0.736391,0.015742,59597,238499,9,55
6,0.200000,0.236534,0.000242,0.859375,0.000483,0.736391,0.015742,70470,227626,9,55
7,0.300000,0.264673,0.000251,0.859375,0.000501,0.736391,0.015742,78860,219236,9,55
8,0.400000,0.290072,0.000255,0.843750,0.000510,0.736391,0.015742,86434,211662,10,54
9,0.500000,0.316256,0.000265,0.843750,0.000529,0.736391,0.015742,94241,203855,10,54


In [17]:
# =========================
# PREDICTION RESULT TABLE
# =========================

test_results = pd.DataFrame({
    "true_label": test_labels.astype(int),
    "probability": test_probs,
    "prediction": (test_probs >= BEST_THRESHOLD).astype(int),
})

test_with_pred = pd.concat([test_df.reset_index(drop=True), test_results], axis=1)
candidate_fraud_full = test_with_pred[test_with_pred["prediction"] == 1].copy()

print("test_with_pred:", test_with_pred.shape)
print("candidate_fraud_full:", candidate_fraud_full.shape)
print("Total real fraud in test:", int(test_with_pred["true_label"].sum()))

test_with_pred.to_csv(OUTPUT_METRICS_DIR / "graph_class_weight_upgrade_no_leakage_test_predictions.csv", index=False)
candidate_fraud_full.to_csv(OUTPUT_METRICS_DIR / "graph_class_weight_upgrade_no_leakage_candidate_fraud.csv", index=False)

test_with_pred: (298160, 47)
candidate_fraud_full: (150524, 47)
Total real fraud in test: 64


In [18]:
# =========================
# TOP-K RANKING + LIFT VS RANDOM
# =========================
# Ini metric paling penting untuk research fraud candidate prioritization.

topk_results = []
total_real_fraud = int(test_with_pred["true_label"].sum())
total_test = len(test_with_pred)
fraud_rate = total_real_fraud / total_test if total_test > 0 else 0

k_values = [50, 100, 250, 500, 1000, 5000, 10000, 20000, 50000]
k_values = [k for k in k_values if k <= total_test]
ranked = test_with_pred.sort_values("probability", ascending=False).reset_index(drop=True)

for k in k_values:
    topk = ranked.head(k)
    tp = int((topk["true_label"] == 1).sum())
    fp = int((topk["true_label"] == 0).sum())
    fn = total_real_fraud - tp
    precision = tp / k if k > 0 else 0
    recall = tp / total_real_fraud if total_real_fraud > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    expected_random_tp = k * fraud_rate
    lift = tp / expected_random_tp if expected_random_tp > 0 else 0
    threshold_at_k = float(topk["probability"].min()) if len(topk) > 0 else np.nan

    topk_results.append({
        "top_k": k,
        "threshold_at_k": threshold_at_k,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "expected_random_tp": expected_random_tp,
        "lift_vs_random": lift,
    })

topk_df = pd.DataFrame(topk_results)
topk_df.to_csv(OUTPUT_METRICS_DIR / "graph_class_weight_upgrade_no_leakage_topk_lift.csv", index=False)
topk_df

,top_k,threshold_at_k,tp,fp,fn,precision,recall,f1,expected_random_tp,lift_vs_random
0,50,1.000000,0,50,64,0.00000,0.000000,0.000000,0.010732,0.000000
1,100,1.000000,3,97,61,0.03000,0.046875,0.036585,0.021465,139.762500
2,250,1.000000,11,239,53,0.04400,0.171875,0.070064,0.053662,204.985000
3,500,1.000000,14,486,50,0.02800,0.218750,0.049645,0.107325,130.445000
4,1000,0.999999,19,981,45,0.01900,0.296875,0.035714,0.214650,88.516250
5,5000,0.999983,29,4971,35,0.00580,0.453125,0.011453,1.073249,27.020750
6,10000,0.999936,30,9970,34,0.00300,0.468750,0.005962,2.146499,13.976250
7,20000,0.999727,34,19966,30,0.00170,0.531250,0.003389,4.292997,7.919875
8,50000,0.997723,36,49964,28,0.00072,0.562500,0.001438,10.732493,3.354300


In [19]:
# =========================
# STAGE-2 RULE FILTERING ANALYSIS ONLY
# =========================
# Rule columns TIDAK dipakai sebagai fitur model.
# Di sini rule dipakai hanya setelah model membuat candidate, sebagai analisis stage-2 filtering.

filter_results = []
total_real_fraud = int(test_with_pred["true_label"].sum())

filter_sets = [("model_only", candidate_fraud_full)]

if "wash_score" in candidate_fraud_full.columns:
    filter_sets.extend([
        ("model + wash_score >= 1", candidate_fraud_full[candidate_fraud_full["wash_score"] >= 1]),
        ("model + wash_score >= 2", candidate_fraud_full[candidate_fraud_full["wash_score"] >= 2]),
        ("model + wash_score >= 3", candidate_fraud_full[candidate_fraud_full["wash_score"] >= 3]),
    ])

if "confidence_category" in candidate_fraud_full.columns:
    filter_sets.extend([
        ("model + confidence medium/high", candidate_fraud_full[candidate_fraud_full["confidence_category"].isin(["medium", "high"])]),
        ("model + confidence high", candidate_fraud_full[candidate_fraud_full["confidence_category"] == "high"]),
    ])

for rule_name, filtered_df in filter_sets:
    tp = int((filtered_df["true_label"] == 1).sum())
    fp = int((filtered_df["true_label"] == 0).sum())
    fn = total_real_fraud - tp
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / total_real_fraud if total_real_fraud > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    filter_results.append({
        "filter": rule_name,
        "candidate_count": len(filtered_df),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    })

filter_df = pd.DataFrame(filter_results)
filter_df.to_csv(OUTPUT_METRICS_DIR / "graph_class_weight_upgrade_no_leakage_stage2_filtering.csv", index=False)
filter_df

,filter,candidate_count,tp,fp,fn,precision,recall,f1
0,model_only,150524,47,150477,17,0.000312,0.734375,0.000624
1,model + wash_score >= 1,47,47,0,17,1.000000,0.734375,0.846847
2,model + wash_score >= 2,47,47,0,17,1.000000,0.734375,0.846847
3,model + wash_score >= 3,4,4,0,60,1.000000,0.062500,0.117647
4,model + confidence medium/high,0,0,0,64,0.000000,0.000000,0.000000
5,model + confidence high,0,0,0,64,0.000000,0.000000,0.000000


In [20]:
# =========================
# FINAL TEST RESULT
# =========================

print("=" * 60)
print("FINAL TEST RESULT")
print("=" * 60)

display(pd.DataFrame([final_result]))

FINAL TEST RESULT


,model,best_epoch_from_validation,pos_weight_mode,raw_pos_weight,used_pos_weight,threshold_strategy,best_threshold_from_validation,validation_fp_budget,accuracy,precision,recall,f1,roc_auc,pr_auc,tn,fp,fn,tp
0,graph_class_weight_upgrade_no_leakage,27,sqrt_capped,50.0,7.071068,fp_budget_then_recall,0.901016,7451,0.495258,0.000312,0.734375,0.000624,0.736391,0.015742,147619,150477,17,47
